# Notebook 2 — Descarga de Niveles de Energía
## Fuente: NIST Atomic Spectra Database (ASD)
## Proyecto: Óptica y Fotónica — Primer Parcial

**Estudiante:** Perez Criollo Andres David  
**Fuente oficial:** https://physics.nist.gov/PhysRefData/ASD/levels_form.html  
**Endpoint API:** http://physics.nist.gov/cgi-bin/ASD/energy1.pl  
**Fecha de descarga:** (completar al ejecutar)  

---

### ¿Qué se descarga en este notebook?

Los **niveles de energía** de cada espectro (elemento + estado de ionización)  
de los primeros 10 elementos de la tabla periódica.

El formulario de niveles solo acepta **un espectro a la vez**, por eso  
este notebook hace **18 peticiones automáticas** (una por cada espectro)  
y al final las une en un solo archivo CSV.

Cada fila del resultado representa un **nivel de energía** y contiene:
- Energía del nivel (en eV)
- Configuración electrónica principal
- Término espectroscópico
- Números cuánticos J y g (degeneración)
- Incertidumbre del nivel
- Factor de Landé-g
- Porcentajes de composición (leading percentages)
- Referencias bibliográficas

---

### Instrucciones
Ejecuta las celdas **en orden de arriba hacia abajo**.  
Al finalizar quedarán guardados:
- Un archivo por espectro en `datos_originales/niveles/`
- Un archivo consolidado `datos_originales/nist_levels_todos_original.csv`  

**No modificar ninguno de esos archivos.**

---
## Celda 1 — Instalación de librerías

In [ ]:
# Instalacion de librerias necesarias
# pandas  -> manejo y analisis de datos tabulares
# requests -> hacer peticiones HTTP a la API del NIST

!pip install pandas requests

---
## Celda 2 — Importación de librerías

In [ ]:
import requests          # Para hacer las peticiones HTTP a la API del NIST
import pandas as pd      # Para leer los CSV y manipular los datos
from io import StringIO  # Para convertir texto de respuesta en objeto legible por pandas
import os               # Para crear carpetas en el sistema de archivos
import time             # Para pausas entre peticiones (buena practica con APIs)
from datetime import datetime  # Para registrar la fecha de descarga

print("Librerias importadas correctamente.")
print(f"Fecha y hora de ejecucion: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## Celda 3 — Crear carpetas de destino

In [ ]:
# Carpeta principal de datos originales
os.makedirs("datos_originales", exist_ok=True)

# Subcarpeta especifica para los niveles de energia (un archivo por espectro)
os.makedirs("datos_originales/niveles", exist_ok=True)

print("Carpetas creadas:")
print("  datos_originales/")
print("  datos_originales/niveles/")

---
## Celda 4 — Definición de la URL y lista de espectros a descargar

In [ ]:
# URL del endpoint de la API del NIST ASD para niveles de energia
# Es diferente al de lineas espectrales (energy1.pl vs lines1.pl)
URL = "http://physics.nist.gov/cgi-bin/ASD/energy1.pl"

# Lista de todos los espectros a descargar
# Formato: 'SIMBOLO ESTADO' donde estado I = neutro, II = primer ionizado
# El formulario de niveles solo acepta UN espectro por peticion
# por eso se hace una peticion separada para cada uno
ESPECTRA = [
    "H I",    # Hidrogeno neutro
    "He I",   # Helio neutro
    "Li I",   # Litio neutro
    "Li II",  # Litio primer ionizado
    "Be I",   # Berilio neutro
    "Be II",  # Berilio primer ionizado
    "B I",    # Boro neutro
    "B II",   # Boro primer ionizado
    "C I",    # Carbono neutro
    "C II",   # Carbono primer ionizado
    "N I",    # Nitrogeno neutro
    "N II",   # Nitrogeno primer ionizado
    "O I",    # Oxigeno neutro
    "O II",   # Oxigeno primer ionizado
    "F I",    # Fluor neutro
    "F II",   # Fluor primer ionizado
    "Ne I",   # Neon neutro
    "Ne II",  # Neon primer ionizado
]

print(f"Total de espectros a descargar: {len(ESPECTRA)}")
print("Lista de espectros:")
for esp in ESPECTRA:
    print(f"  - {esp}")

---
## Celda 5 — Definición de parámetros base de la consulta

Cada parámetro corresponde exactamente a una opción del formulario web en:  
https://physics.nist.gov/PhysRefData/ASD/levels_form.html

In [ ]:
# Parametros base que se usaran para TODOS los espectros
# Solo cambiara el campo 'spectrum' en cada peticion

PARAMETROS_BASE = {

    # --- ESPECTRO (se reemplazara en cada iteracion) ---
    # "spectrum": se asigna dinamicamente en el loop

    # --- UNIDADES DE ENERGIA ---
    "units": 1,
    # 0 = cm-1 (numero de onda)
    # 1 = eV (electronvoltios)  <-- SELECCIONADO
    # 2 = Rydberg
    # 3 = Hartree
    # 4 = GHz
    # Se elige eV porque es la unidad mas universal en fisica moderna
    # y la mas intuitiva para comparar entre elementos

    # --- FORMATO DE SALIDA ---
    "format": 2,
    # 0 = HTML
    # 1 = ASCII
    # 2 = CSV  <-- SELECCIONADO
    # 3 = Tab-delimited

    # --- MOSTRAR RESULTADO COMPLETO ---
    "output": 0,
    # 0 = Todo en una sola pagina (in its entirety)  <-- SELECCIONADO
    # 1 = Paginado

    # --- ORDENAMIENTO ---
    "order": 0,
    # 0 = Ordenado por energia (Energy ordered)  <-- SELECCIONADO
    # 1 = Ordenado por termino (Term ordered)

    # --- INFORMACION DE NIVEL: CONFIGURACION PRINCIPAL ---
    "show_conf": 1,
    # 1 = Incluir la configuracion electronica principal
    # Ejemplo: 1s2 2s2 2p1
    # Necesaria para identificar cada nivel en el modelo relacional

    # --- INFORMACION DE NIVEL: TERMINO ESPECTROSCOPICO ---
    "show_term": 1,
    # 1 = Incluir el termino espectroscopico (ej. 2P*, 3D, 1S)
    # Describe el estado cuantico completo del nivel atomico

    # --- INFORMACION DE NIVEL: INCERTIDUMBRE ---
    "show_level_unc": 1,
    # 1 = Incluir la incertidumbre de la energia del nivel
    # Indica la calidad y precision del dato experimental

    # --- INFORMACION DE NIVEL: NUMERO CUANTICO J ---
    "show_j": 1,
    # 1 = Incluir el numero cuantico de momento angular total J
    # Necesario para calcular la degeneracion: g = 2J + 1

    # --- INFORMACION DE NIVEL: DEGENERACION g ---
    "show_g": 1,
    # 1 = Incluir la degeneracion estadistica g del nivel
    # g = 2J + 1, aparece en el calculo de la fuerza de oscilador

    # --- INFORMACION DE NIVEL: IDs DE NIVEL ---
    "show_level_id": 1,
    # 1 = Incluir los identificadores internos del nivel en la BD del NIST
    # Util para hacer JOIN con la tabla de lineas espectrales

    # --- FACTOR DE LANDE-g ---
    "show_lande_g": 1,
    # 1 = Incluir el factor de Lande-g del nivel
    # Describe como responde el nivel a campos magneticos externos
    # Relevante para efectos Zeeman y aplicaciones en fotonica cuantica

    # --- PORCENTAJES DE COMPOSICION (Leading percentages) ---
    "show_perc": 1,
    # 1 = Incluir los porcentajes de composicion de la funcion de onda
    # Indica que fraccion del nivel corresponde a cada configuracion
    # Util para caracterizar niveles mixtos en atomos multielectronicos

    # --- REFERENCIAS BIBLIOGRAFICAS ---
    "biblio": 1,
    # 1 = Incluir referencias de la fuente de cada nivel de energia
    # Documenta la procedencia cientifica — requerido por la rubrica

    # --- PARAMETROS TECNICOS ---
    "submit": "Retrieve Data",
    "page_size": 15,
    "temp": "",    # Temperatura para funcion de particion (no requerida)
}

print("Parametros base definidos correctamente.")
print(f"Unidades de energia: eV")
print(f"Formato de salida: CSV")
print(f"Columnas activadas: configuracion, termino, incertidumbre, J, g, Level IDs, Lande-g, leading percentages, referencias")

---
## Celda 6 — Función auxiliar para limpiar y parsear la respuesta del NIST

In [ ]:
def parsear_respuesta_nist(texto_respuesta, nombre_espectro):
    """
    Convierte el texto CSV del NIST en un DataFrame de pandas.
    Agrega una columna 'espectro' para identificar el elemento
    despues de consolidar todos los archivos en uno solo.
    
    Parametros:
        texto_respuesta (str): Texto crudo devuelto por la API del NIST
        nombre_espectro (str): Nombre del espectro consultado (ej. 'H I')
    
    Retorna:
        pd.DataFrame o None si hubo error
    """
    try:
        # Filtrar lineas vacias y separadores del NIST
        lineas = texto_respuesta.split("\n")
        lineas_utiles = [
            linea for linea in lineas
            if linea.strip() and not linea.startswith("---")
        ]
        texto_limpio = "\n".join(lineas_utiles)
        
        # Parsear el CSV
        df = pd.read_csv(
            StringIO(texto_limpio),
            sep=",",
            low_memory=False,
            on_bad_lines="warn"
        )
        
        # Agregar columna identificadora del espectro
        # Esto es fundamental para poder distinguir los niveles
        # cuando se consoliden todos los archivos en uno solo
        df.insert(0, "espectro", nombre_espectro)
        
        return df
    
    except Exception as e:
        print(f"  ERROR al parsear {nombre_espectro}: {e}")
        return None

print("Funcion auxiliar definida.")

---
## Celda 7 — Descarga automática de todos los espectros

Esta celda hace 18 peticiones al NIST, una por cada espectro.  
Guarda cada resultado como archivo individual y al final los consolida en uno solo.

In [ ]:
# Lista para acumular todos los DataFrames descargados
lista_dfs = []

# Registro de resultados de cada descarga
registro = []

print("Iniciando descarga de niveles de energia...")
print(f"Total de espectros: {len(ESPECTRA)}")
print("=" * 60)

for i, espectro in enumerate(ESPECTRA, start=1):
    
    print(f"\n[{i:02d}/{len(ESPECTRA)}] Descargando: {espectro} ...")
    
    # Construir parametros para esta peticion especifica
    # Copiamos los parametros base y agregamos el espectro
    parametros = PARAMETROS_BASE.copy()
    parametros["spectrum"] = espectro
    
    try:
        # Hacer la peticion HTTP al NIST
        response = requests.get(URL, params=parametros, timeout=60)
        
        # Verificar codigo de respuesta
        if response.status_code != 200:
            print(f"  ERROR HTTP {response.status_code} para {espectro}")
            registro.append({"espectro": espectro, "estado": "ERROR HTTP", "filas": 0})
            continue
        
        # Guardar el texto crudo del NIST como archivo original individual
        # Nombre de archivo: reemplazar espacios por guion bajo
        nombre_archivo = espectro.replace(" ", "").replace("I", "I").lower()
        nombre_archivo = f"datos_originales/niveles/nist_levels_{espectro.replace(' ', '_')}_original.csv"
        
        with open(nombre_archivo, "w", encoding="utf-8") as f:
            f.write(response.text)
        
        # Parsear la respuesta a DataFrame
        df = parsear_respuesta_nist(response.text, espectro)
        
        if df is not None and len(df) > 0:
            lista_dfs.append(df)
            print(f"  OK — {len(df):,} niveles descargados — guardado en {nombre_archivo}")
            registro.append({"espectro": espectro, "estado": "OK", "filas": len(df)})
        else:
            print(f"  AVISO — Respuesta vacia o sin datos para {espectro}")
            registro.append({"espectro": espectro, "estado": "VACIO", "filas": 0})
    
    except requests.exceptions.Timeout:
        print(f"  ERROR — Tiempo de espera agotado para {espectro}")
        registro.append({"espectro": espectro, "estado": "TIMEOUT", "filas": 0})
    
    except Exception as e:
        print(f"  ERROR inesperado para {espectro}: {e}")
        registro.append({"espectro": espectro, "estado": f"ERROR: {e}", "filas": 0})
    
    # Pausa de 1 segundo entre peticiones
    # Buena practica para no saturar el servidor del NIST
    time.sleep(1)

print("\n" + "=" * 60)
print("DESCARGA COMPLETADA")
print("=" * 60)

# Mostrar resumen de descargas
df_registro = pd.DataFrame(registro)
print(df_registro.to_string(index=False))
print(f"\nTotal espectros exitosos: {df_registro[df_registro['estado']=='OK'].shape[0]}/{len(ESPECTRA)}")

---
## Celda 8 — Consolidar todos los espectros en un solo DataFrame

In [ ]:
# Unir todos los DataFrames individuales en uno solo
# ignore_index=True renumera los indices de 0 a N

if lista_dfs:
    df_levels = pd.concat(lista_dfs, ignore_index=True)
    
    print("DataFrame consolidado creado.")
    print(f"Total de filas:    {df_levels.shape[0]:,}")
    print(f"Total de columnas: {df_levels.shape[1]}")
    print(f"\nNombres de columnas:")
    for col in df_levels.columns:
        print(f"  - {col}")
else:
    print("ERROR: No se pudo descargar ningun espectro. Revisar conexion a internet.")

---
## Celda 9 — Inspección inicial del DataFrame consolidado

In [ ]:
# Primeras filas del DataFrame consolidado
print("=== Primeras 5 filas del dataset consolidado ===")
df_levels.head()

In [ ]:
# Cuantos niveles hay por espectro
print("=== Niveles de energia por espectro ===")
conteo = df_levels.groupby("espectro").size().reset_index(name="num_niveles")
print(conteo.to_string(index=False))

In [ ]:
# Tipos de datos
print("=== Tipos de datos por columna ===")
print(df_levels.dtypes)

In [ ]:
# Resumen estadistico
print("=== Resumen estadistico ===")
df_levels.describe(include="all")

---
## Celda 10 — Guardar el archivo consolidado original

In [ ]:
# Nombre del archivo consolidado
NOMBRE_CONSOLIDADO = "datos_originales/nist_levels_todos_original.csv"

# Guardar el DataFrame consolidado como CSV
# index=False evita guardar el numero de fila como columna extra
df_levels.to_csv(NOMBRE_CONSOLIDADO, index=False, encoding="utf-8")

print(f"Archivo consolidado guardado en: {NOMBRE_CONSOLIDADO}")
print(f"Tamano del archivo: {os.path.getsize(NOMBRE_CONSOLIDADO):,} bytes")
print()
print("IMPORTANTE: Este archivo NO debe modificarse.")
print("Es la evidencia de la fuente de datos oficial (NIST ASD).")
print()
print("=" * 60)
print("DOCUMENTACION DE DESCARGA")
print("=" * 60)
print(f"Fuente:            NIST Atomic Spectra Database (ASD)")
print(f"URL base:          {URL}")
print(f"Tipo de dato:      Niveles de energia atomicos")
print(f"Elementos:         H, He, Li, Be, B, C, N, O, F, Ne (neutros e ionizados)")
print(f"Unidades energia:  eV (electronvoltios)")
print(f"Formato salida:    CSV")
print(f"Fecha descarga:    {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Espectros OK:      {df_registro[df_registro['estado']=='OK'].shape[0]}/{len(ESPECTRA)}")
print(f"Total filas:       {df_levels.shape[0]:,}")
print(f"Total columnas:    {df_levels.shape[1]}")
print(f"Archivo:           {NOMBRE_CONSOLIDADO}")
print(f"Archivos individuales en: datos_originales/niveles/")

---
## Celda 11 — Verificar la estructura de archivos generados

In [ ]:
# Listar todos los archivos en la carpeta datos_originales
print("=== Archivos en datos_originales/ ===")

for raiz, carpetas, archivos in os.walk("datos_originales"):
    nivel = raiz.replace("datos_originales", "").count(os.sep)
    sangria = "  " * nivel
    print(f"{sangria}{os.path.basename(raiz)}/")
    sub_sangria = "  " * (nivel + 1)
    for archivo in sorted(archivos):
        ruta = os.path.join(raiz, archivo)
        tamano = os.path.getsize(ruta)
        print(f"{sub_sangria}{archivo}  ({tamano:,} bytes)")

---
## ✅ Descarga completada

Los archivos generados son:

| Archivo | Contenido |
|---|---|
| `datos_originales/nist_levels_todos_original.csv` | Todos los niveles consolidados |
| `datos_originales/niveles/nist_levels_H_I_original.csv` | Solo niveles de H I |
| `datos_originales/niveles/nist_levels_He_I_original.csv` | Solo niveles de He I |
| `datos_originales/niveles/...` | Un archivo por espectro |

**Siguiente paso:** Notebook de exploración y limpieza con pandas,  
usando los archivos:
- `datos_originales/nist_lines_H_Ne_original.csv` (del Notebook 1)
- `datos_originales/nist_levels_todos_original.csv` (de este Notebook 2)